# 02b — Rebuild BIO tags from character offsets

The original BIO conversion matched entities by whitespace-token equality. The
projection stores **stems** (`Aizawl`) while sentences contain **inflected
tokens** (`Aizawlah`), so suffixed entities received no tag. Combined with
`break`-after-first-match collapsing repeated entities, this dropped **20.5%**
of projected entities from training.

This notebook rebuilds the tags using character-offset **overlap**, keeps the
original train/dev/test sentence partitions, and reports what changed.

**Run from the repository root**, not from inside `notebooks/`.

In [1]:
from pathlib import Path
import json, sys
from collections import Counter, defaultdict

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    print(f"Stepped up to repo root: {ROOT}")

PROC = ROOT / "data" / "processed"
CORPUS   = PROC / "mizo_ner_annotations_dedup_aggressive.jsonl"
OLD_BIO  = PROC / "mizo_ner_bio.json"
OLD_SPLIT = {s: PROC / f"mizo_ner_{s}.json" for s in ("train", "dev", "test")}

for p in [CORPUS, OLD_BIO, *OLD_SPLIT.values()]:
    print(("  ok   " if p.exists() else "  MISS ") + str(p.relative_to(ROOT)))
    if not p.exists():
        sys.exit("Missing input - check data/processed/")

Stepped up to repo root: C:\Users\Haulai\mizo-ner
  ok   data\processed\mizo_ner_annotations_dedup_aggressive.jsonl
  ok   data\processed\mizo_ner_bio.json
  ok   data\processed\mizo_ner_train.json
  ok   data\processed\mizo_ner_dev.json
  ok   data\processed\mizo_ner_test.json


## Cell 2: Load the corpus

In [2]:
records = []
with open(CORPUS, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Sentences        : {len(records):,}")
print(f"Entity spans     : {sum(len(r['entities']) for r in records):,}")
print(f"Keys present     : {list(records[0].keys())}")
print()
print("First record:")
print(f"  text     : {records[0]['text']}")
print(f"  entities : {records[0]['entities']}")

Sentences        : 441,178
Entity spans     : 590,655
Keys present     : ['text', 'entities', 'english_source']

First record:
  text     : Kar thum chawlh Liana-an a la.
  entities : [[16, 21, 'PERSON']]


## Cell 3: Confirm alignment between jsonl line order and the old `id`

The old BIO file was built from an Excel export that suffered a UTF-8 to
latin-1 mangling, so its text carries mojibake (`ṭha` became `á¹­ha`) even
though the ids are correct. We therefore compare on a mangling-tolerant basis:
a sentence counts as aligned if it matches the jsonl text either exactly or
after applying the same corruption.

We only need the ids to be trustworthy, since they are used solely to reuse the
original train/dev/test partitioning. The rebuilt tags take their text from the
jsonl, which is clean.


In [3]:
old_bio = json.load(open(OLD_BIO, encoding="utf-8"))
print(f"Old BIO sentences: {len(old_bio):,}")

by_id = {r["id"]: r for r in old_bio}

def mangle(s):
    """Reproduce the UTF-8 -> latin-1 corruption present in the Excel export."""
    try:
        return s.encode("utf-8").decode("latin-1")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

exact = mojibake = neither = 0
for lineno, rec in enumerate(records, start=1):
    if lineno not in by_id:
        continue
    old_txt = " ".join(by_id[lineno]["tokens"])
    new_txt = " ".join(rec["text"].split())
    if old_txt == new_txt:
        exact += 1
    elif old_txt == " ".join(mangle(new_txt).split()):
        mojibake += 1
    else:
        neither += 1
    if lineno >= 20000:
        break

tot = exact + mojibake + neither
aligned = (exact + mojibake) / tot * 100
print(f"  exact match    : {exact:>7,}  ({exact/tot*100:5.2f}%)")
print(f"  mojibake match : {mojibake:>7,}  ({mojibake/tot*100:5.2f}%)")
print(f"  neither        : {neither:>7,}  ({neither/tot*100:5.2f}%)")
print(f"  ALIGNED        : {aligned:5.2f}%")

assert aligned > 98, "Line order does not match id - stop and investigate"
print("\nConfirmed: jsonl line number == old BIO id.")
print("Rebuilt tags will use the clean jsonl text, correcting the mojibake.")


Old BIO sentences: 441,177
  exact match    :  17,099  (85.50%)
  mojibake match :   2,757  (13.79%)
  neither        :     144  ( 0.72%)
  ALIGNED        : 99.28%

Confirmed: jsonl line number == old BIO id.
Rebuilt tags will use the clean jsonl text, correcting the mojibake.


## Cell 4: Offset-based BIO conversion

Whitespace tokens carry character spans. A token is tagged when its span
**overlaps** the entity span, so `Aizawlah` is tagged for an entity marked at
`Aizawl`. Every entity is placed independently, so repeated mentions all survive.

In [4]:
def tokens_with_spans(text):
    toks, i = [], 0
    for t in text.split():
        start = text.index(t, i)
        toks.append((t, start, start + len(t)))
        i = start + len(t)
    return toks

def bio_from_offsets(text, entities):
    toks = tokens_with_spans(text)
    tags = ["O"] * len(toks)
    placed = 0
    for ent in sorted(entities, key=lambda e: (e[0], -(e[1] - e[0]))):
        s, e, label = ent[0], ent[1], ent[2]
        hit = [i for i, (_, ts, te) in enumerate(toks) if ts < e and te > s]
        if not hit:
            continue
        if any(tags[i] != "O" for i in hit):     # already covered; don't overwrite
            continue
        tags[hit[0]] = f"B-{label}"
        for i in hit[1:]:
            tags[i] = f"I-{label}"
        placed += 1
    return [t for t, _, _ in toks], tags, placed

# sanity check on the motivating example
demo = {"text": "Aizawlah an thuthmun tur ngaihtuah mek a ni.",
        "entities": [[0, 6, "GPE"]]}
tk, tg, _ = bio_from_offsets(demo["text"], demo["entities"])
print("Demo:", list(zip(tk, tg))[:3])
assert tg[0] == "B-GPE", "offset overlap failed"
print("Offset matching works on the suffixed case.")

Demo: [('Aizawlah', 'B-GPE'), ('an', 'O'), ('thuthmun', 'O')]
Offset matching works on the suffixed case.


## Cell 5: Convert the whole corpus

In [5]:
new_bio, total_placed, total_avail = [], 0, 0
for lineno, rec in enumerate(records, start=1):
    ents = rec.get("entities", [])
    total_avail += len(ents)
    toks, tags, placed = bio_from_offsets(rec["text"], ents)
    total_placed += placed
    if not toks:
        continue
    new_bio.append({"id": lineno, "tokens": toks, "tags": tags})
    if lineno % 100000 == 0:
        print(f"  {lineno:,} / {len(records):,}")

print(f"\nSentences written : {len(new_bio):,}")
print(f"Entities available: {total_avail:,}")
print(f"Entities tagged   : {total_placed:,}  ({total_placed/total_avail*100:.1f}%)")

  100,000 / 441,178
  200,000 / 441,178
  300,000 / 441,178
  400,000 / 441,178

Sentences written : 441,178
Entities available: 590,655
Entities tagged   : 586,552  (99.3%)


## Cell 6: Old versus new, per entity type

In [6]:
def b_counts(bio):
    c = Counter()
    for r in bio:
        for t in r["tags"]:
            if t.startswith("B-"):
                c[t[2:]] += 1
    return c

old_c, new_c = b_counts(old_bio), b_counts(new_bio)
corpus_c = Counter(e[2] for r in records for e in r.get("entities", []))

print(f"{'Entity':<14}{'corpus':>10}{'old BIO':>10}{'new BIO':>10}{'recovered':>11}{'now kept':>10}")
print("-" * 65)
for lab, tot in corpus_c.most_common():
    o, n = old_c.get(lab, 0), new_c.get(lab, 0)
    print(f"{lab:<14}{tot:>10,}{o:>10,}{n:>10,}{n-o:>+11,}{n/tot*100:>9.1f}%")

to, tn, tt = sum(old_c.values()), sum(new_c.values()), sum(corpus_c.values())
print("-" * 65)
print(f"{'TOTAL':<14}{tt:>10,}{to:>10,}{tn:>10,}{tn-to:>+11,}{tn/tt*100:>9.1f}%")
print(f"\nCoverage: {to/tt*100:.1f}%  ->  {tn/tt*100:.1f}%")

Entity            corpus   old BIO   new BIO  recovered  now kept
-----------------------------------------------------------------
PERSON           326,978   258,439   325,709    +67,270     99.6%
GPE              116,495    83,462   115,177    +31,715     98.9%
ORG              104,485    90,146   103,370    +13,224     98.9%
NORP              19,726    18,561    19,510       +949     98.9%
LOC                6,857     5,867     6,826       +959     99.5%
LANGUAGE           4,655     4,410     4,614       +204     99.1%
WORK_OF_ART        3,886     2,788     3,815     +1,027     98.2%
FAC                3,174     1,958     3,172     +1,214     99.9%
PRODUCT            2,934     2,564     2,896       +332     98.7%
EVENT                884       744       882       +138     99.8%
LAW                  581       510       581        +71    100.0%
-----------------------------------------------------------------
TOTAL            590,655   469,449   586,552   +117,103     99.3%

Coverage:

## Cell 7: Reapply the original splits

Sentence partitions are taken from the saved split files, so the only thing that
changes between the old and new experiments is the tagging.

In [7]:
split_of = {}
for name, path in OLD_SPLIT.items():
    for r in json.load(open(path, encoding="utf-8")):
        split_of[r["id"]] = name

print(f"Split assignment covers {len(split_of):,} sentence ids")

buckets = defaultdict(list)
unassigned = 0
for r in new_bio:
    s = split_of.get(r["id"])
    if s is None:
        unassigned += 1
    else:
        buckets[s].append(r)

for name in ("train", "dev", "test"):
    print(f"  {name:<6}{len(buckets[name]):>8,}")
print(f"  unassigned{unassigned:>6,}   (expected 0-1)")

Split assignment covers 441,177 sentence ids
  train  352,941
  dev     44,118
  test    44,118
  unassigned     1   (expected 0-1)


## Cell 8: Write the new files

In [8]:
out = PROC / "bio_v2"
out.mkdir(exist_ok=True)

json.dump(new_bio, open(out / "mizo_ner_bio.json", "w", encoding="utf-8"), ensure_ascii=False)
for name in ("train", "dev", "test"):
    json.dump(buckets[name], open(out / f"mizo_ner_{name}.json", "w", encoding="utf-8"),
              ensure_ascii=False)

stats = {
    "sentences": len(new_bio),
    "entities_in_corpus": int(sum(corpus_c.values())),
    "entities_tagged_old": int(sum(old_c.values())),
    "entities_tagged_new": int(sum(new_c.values())),
    "coverage_old_pct": round(sum(old_c.values()) / sum(corpus_c.values()) * 100, 2),
    "coverage_new_pct": round(sum(new_c.values()) / sum(corpus_c.values()) * 100, 2),
    "per_type": {k: {"corpus": int(v),
                     "old": int(old_c.get(k, 0)),
                     "new": int(new_c.get(k, 0))} for k, v in corpus_c.items()},
    "splits": {k: len(v) for k, v in buckets.items()},
}
json.dump(stats, open(ROOT / "results" / "ner" / "bio_v2_stats.json", "w"), indent=2)

print(f"Written to {out.relative_to(ROOT)}:")
for p in sorted(out.iterdir()):
    print(f"  {p.name:<26}{p.stat().st_size/1024**2:>8.1f} MB")
print("\nStats -> results/ner/bio_v2_stats.json")

Written to data\processed\bio_v2:
  mizo_ner_bio.json             82.9 MB
  mizo_ner_dev.json              8.3 MB
  mizo_ner_test.json             8.3 MB
  mizo_ner_train.json           66.3 MB

Stats -> results/ner/bio_v2_stats.json


## Cell 9: Eyeball the difference

Sentences whose tagging changed, so you can confirm the new tags are right
rather than merely more numerous.

In [9]:
old_by_id = {r["id"]: r for r in old_bio}
shown = 0
for r in new_bio:
    o = old_by_id.get(r["id"])
    if not o or o["tags"] == r["tags"]:
        continue
    print(f"--- id {r['id']} ---")
    for i, tok in enumerate(r["tokens"]):
        ot = o["tags"][i] if i < len(o["tags"]) else "?"
        nt = r["tags"][i]
        flag = "  <-- CHANGED" if ot != nt else ""
        if ot != "O" or nt != "O":
            print(f"  {tok:<22}{ot:<14}->  {nt:<14}{flag}")
    print()
    shown += 1
    if shown >= 12:
        break

--- id 1 ---
  Liana-an              O             ->  B-PERSON        <-- CHANGED

--- id 3 ---
  Aizawlah              O             ->  B-GPE           <-- CHANGED

--- id 6 ---
  Alvin-an              O             ->  B-PERSON        <-- CHANGED

--- id 13 ---
  VL.                   O             ->  B-GPE           <-- CHANGED

--- id 14 ---
  Hminga                B-PERSON      ->  B-PERSON      
  Mawii-in              O             ->  B-PERSON        <-- CHANGED

--- id 16 ---
  Boston-ah             O             ->  B-GPE           <-- CHANGED

--- id 17 ---
  Bibleah               O             ->  B-WORK_OF_ART   <-- CHANGED

--- id 19 ---
  Labana                O             ->  B-ORG           <-- CHANGED

--- id 20 ---
  Kerala-in             O             ->  B-GPE           <-- CHANGED

--- id 21 ---
  Dinga'n               O             ->  B-PERSON        <-- CHANGED

--- id 31 ---
  Samu-a                O             ->  B-PERSON        <-- CHANGED

--- id 35 -